In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Esegui Circuit dinamici

import Tabs from '@theme/Tabs';
import TabItem from '@theme/TabItem';



<Accordion>
<AccordionItem title="Versioni dei pacchetti">

Il codice in questa pagina è stato sviluppato con i seguenti requisiti.
Ti consigliamo di utilizzare queste versioni o versioni più recenti.

```
qiskit[all]~=2.4.0
qiskit-ibm-runtime~=0.46.1
```
</AccordionItem>
</Accordion>
I Circuit dinamici sono strumenti potenti con cui puoi misurare i qubit nel mezzo dell'esecuzione di un Circuit quantistico e poi eseguire operazioni di logica classica all'interno del Circuit, in base al risultato di quelle misurazioni mid-circuit. Questo processo è anche noto come _feedforward classico_. Sebbene questi siano ancora i primi passi per capire come sfruttare al meglio i Circuit dinamici, la comunità di ricerca quantistica ha già identificato una serie di casi d'uso, come i seguenti:

* Preparazione efficiente dello stato quantistico, come lo [stato GHZ](https://journals.aps.org/prxquantum/abstract/10.1103/PRXQuantum.5.030339), lo [stato W](https://arxiv.org/abs/2403.07604) (per ulteriori informazioni sullo stato W, consulta anche ["State preparation by shallow circuits using feed forward"](https://arxiv.org/abs/2307.14840)), e un'ampia classe di [matrix product states](https://arxiv.org/abs/2404.16083)
* [Entanglement a lungo raggio efficiente](https://journals.aps.org/prxquantum/abstract/10.1103/PRXQuantum.5.030339) tra qubit sullo stesso chip usando Circuit superficiali
* [Campionamento efficiente di Circuit simili a IQP](https://arxiv.org/pdf/2505.04705)

Questi miglioramenti apportati dai Circuit dinamici, tuttavia, comportano dei compromessi. Le misurazioni mid-circuit e le operazioni classiche hanno tipicamente tempi di esecuzione più lunghi rispetto ai Gate a due Qubit, e questo aumento del tempo potrebbe annullare i benefici della riduzione della profondità del Circuit. Pertanto, la riduzione della durata della misurazione mid-circuit è un'area di miglioramento su cui IBM Quantum&reg; si concentra con il rilascio della [nuova versione](/guides/latest-updates#new-version-dynamic-circuits) dei Circuit dinamici. Per altre restrizioni quando si usano i Circuit dinamici, consulta la tabella di compatibilità delle funzionalità di [Estimator](/guides/estimator-options#options-compatibility-table) o [Sampler](/guides/sampler-options#options-compatibility-table).

La [specifica OpenQASM 3](https://openqasm.com/language/classical.html#looping-and-branching) definisce una serie di strutture di flusso di controllo, ma Qiskit Runtime attualmente supporta solo l'istruzione condizionale `if`. In Qiskit SDK, questo corrisponde al metodo [if_test](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#if_test) su [QuantumCircuit](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit). Questo metodo restituisce un [context manager](https://docs.python.org/3/reference/datamodel.html#with-statement-context-managers) e viene tipicamente usato in un'istruzione `with`. Questa guida descrive come usare questa istruzione condizionale.

> **Note:** Gli esempi di codice in questa guida usano l'istruzione di misurazione standard per le misurazioni mid-circuit. Tuttavia, si raccomanda di usare invece l'istruzione `MidCircuitMeasure`, se il Backend la supporta. Consulta la [sezione Misurazioni mid-circuit](#midcircuit) per i dettagli.
## Trova i Backend che supportano i Circuit dinamici
Per trovare tutti i Backend a cui il tuo account può accedere e che supportano i Circuit dinamici, esegui codice come il seguente. Questo esempio presuppone che tu abbia [salvato le tue credenziali di accesso](/guides/save-credentials). Potresti anche [specificare esplicitamente le credenziali](/guides/initialize-account#explicit) quando inizializzi il tuo account del servizio Qiskit Runtime. Questo ti permetterebbe di visualizzare i Backend disponibili su un'istanza o tipo di piano specifico, ad esempio.

> **Note:** - I Backend disponibili per l'account dipendono dall'istanza specificata nelle credenziali.
> - La nuova versione dei Circuit dinamici è ora disponibile per tutti gli utenti su tutti i Backend. Consulta [l'annuncio](/guides/latest-updates#new-dynamic-circuits) per ulteriori dettagli.

In [ ]:
# This cell is hidden from users. It hides all those "...instance was not set..." warnings.
import warnings

warnings.filterwarnings("ignore", message=".*Instance was not set*")

In [2]:
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
dc_backends = service.backends(dynamic_circuits=True)
print(dc_backends)

[<IBMBackend('ibm_pittsburgh')>, <IBMBackend('ibm_kingston')>, <IBMBackend('ibm_marrakesh')>, <IBMBackend('ibm_fez')>, <IBMBackend('ibm_boston')>]


<span id="midcircuit"></span>
## Mid-circuit measurements

Prior to `qiskit-ibm-runtime` v0.43.0, `measure` was the only measurement instruction in Qiskit. Mid-circuit measurements, however, have different tuning requirements than _terminal_ measurements (measurements that happen at the end of a circuit). For example, you need to consider the instruction duration when tuning a mid-circuit measurement because longer instructions cause noisier circuits. You don't need to consider instruction duration for terminal measurements because there are no instructions after terminal measurements.

<Admonition type="note">
The `MidCircuitMeasure` instruction maps to the `measure_2` instruction reported in the backend's `supported_instructions`. However,  `measure_2` is not supported on all backends. Use `service.backends(filters=lambda b: "measure_2" in b.supported_instructions)` to find backends that support it.  New measurements might be added in the future, but this is not guarenteed.
</Admonition>

### `MidCircuitMeasure` method

In `qiskit-ibm-runtime` v0.43.0, the `MidCircuitMeasure` instruction was introduced. As the name suggests, it is a new measurement instruction that is optimized for mid-circuit on IBM&reg; QPUs.  While you can use `QuantumCircuit.measure` for a mid-circuit measurement, because of its design, `MidCircuitMeasure` is typically a better choice.  For example, it adds less overhead to your circuit than when using `QuantumCircuit.measure`.

In [3]:
from qiskit import QuantumCircuit
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime.circuit import MidCircuitMeasure

service = QiskitRuntimeService()
backend = service.least_busy(
    operational=True, simulator=False, dynamic_circuits=True
)

circ = QuantumCircuit(2, 2)
circ.x(0)
circ.append(MidCircuitMeasure(), [0], [0])
# circ.measure([0], [0])
# circ.measure_all()
print(circ.draw(cregbundle=False))

     ┌───┐┌────────────┐
q_0: ┤ X ├┤0           ├
     └───┘│            │
q_1: ─────┤  Measure_2 ├
          │            │
c_0: ═════╡0           ╞
          └────────────┘
c_1: ═══════════════════
                        


<span id="midcircuit"></span>

## Misurazioni mid-circuit

Prima di `qiskit-ibm-runtime` v0.43.0, `measure` era l'unica istruzione di misurazione in Qiskit. Le misurazioni mid-circuit, tuttavia, hanno requisiti di tuning diversi rispetto alle misurazioni _terminali_ (misurazioni che avvengono alla fine di un Circuit). Ad esempio, è necessario considerare la durata dell'istruzione durante il tuning di una misurazione mid-circuit perché le istruzioni più lunghe causano Circuit più rumorosi. Non è necessario considerare la durata dell'istruzione per le misurazioni terminali perché non ci sono istruzioni dopo le misurazioni terminali.

> **Note:** L'istruzione `MidCircuitMeasure` mappa all'istruzione `measure_2` riportata nelle `supported_instructions` del Backend. Tuttavia, `measure_2` non è supportata su tutti i Backend. Usa `service.backends(filters=lambda b: "measure_2" in b.supported_instructions)` per trovare i Backend che la supportano. In futuro potrebbero essere aggiunte nuove misurazioni, ma ciò non è garantito.

### Metodo `MidCircuitMeasure`

In `qiskit-ibm-runtime` v0.43.0, è stata introdotta l'istruzione `MidCircuitMeasure`. Come suggerisce il nome, è una nuova istruzione di misurazione ottimizzata per le misurazioni mid-circuit su QPU IBM&reg;. Sebbene tu possa usare `QuantumCircuit.measure` per una misurazione mid-circuit, a causa del suo design, `MidCircuitMeasure` è tipicamente una scelta migliore. Ad esempio, aggiunge meno overhead al tuo Circuit rispetto all'uso di `QuantumCircuit.measure`.

In [4]:
from qiskit_ibm_runtime import SamplerV2, QiskitRuntimeService
from qiskit.circuit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.transpiler import generate_preset_pass_manager

service = QiskitRuntimeService()
backend = service.least_busy(
    operational=True, simulator=False, dynamic_circuits=True
)

# Create a dynamic circuit

qubits = QuantumRegister(1)
clbits = ClassicalRegister(1)
qc = QuantumCircuit(qubits, clbits)
(q0,) = qubits
(c0,) = clbits

qc.h(q0)
qc.measure(q0, c0)
with qc.if_test((c0, 1)):
    qc.x(q0)
qc.measure(q0, c0)


# Convert to an ISA circuit for the given backend

pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(qc)

# Generate samplers for backend targets
sampler = SamplerV2(backend)

# Submit jobs
sampler_job = sampler.run([isa_circuit])
result = sampler_job.result()

print(
    f">>> {' Job ID:':<10}  {sampler_job.job_id()} ({sampler_job.status()})"
)

>>>  Job ID:    d88cakp789is7391vq0g (DONE)


## Qiskit Runtime limitations

Be aware of the following constraints when running dynamic circuits in Qiskit Runtime.

- Due to the limited physical memory on control electronics, there is also a limit on the number of `if` statements and size of their operands. This limit is a function of the number of broadcasts and the number of broadcasted bits in a job (not a circuit).

   When processing an `if` condition, measurement data needs to be transferred to the control logic to make that evaluation. A broadcast is a transfer of unique classical data, and broadcasted bits is the number of classical bits being transferred. Consider the following:

   ```python
   c0 = ClassicalRegister(3)
   c1 = ClassicalRegister(5)
   ...
   with circuit.if_test((c0, 1)) ...
   with circuit.if_test((c0, 3)) ...
   with circuit.if_test((c1[2], 1)) ...
   ```
   In the previous code example, the first two `if_test` objects on `c0` are considered one broadcast because the content of `c0` did not change, and thus does not need to be re-broadcasted. The `if_test` on `c1` is a second broadcast. The first one broadcasts all three bits in `c0` and the second broadcasts just one bit, making a total of four broadcasted bits.

   Currently, if you broadcast 60 bits each time, then the job can have approximately 300 broadcasts. If you broadcast just one bit each time, however, then the job can have 2400 broadcasts.

- The operand used in an `if_test` statement must be 32 or fewer bits. Thus, if you are comparing an entire `ClassicalRegister`, the size of that `ClassicalRegister` must be 32 or fewer bits. If you are comparing just a single bit from a `ClassicalRegister`, however, that `ClassicalRegister` can be of any size (since the operand is only one bit).

   For example, the "Not valid" code block does not work because `cr` is more than 32 bits.  You can, however, use a classical register wider than 32 bits if you are testing only one bit, as shown in the "Valid" code block.

   <Tabs>
   <TabItem value="Not valid" label="Not valid">
      ```python
         cr = ClassicalRegister(50)
         qr = QuantumRegister(50)
         circuit = QuantumCircuit(qr, cr)
         ...
         circ.measure(qr, cr)
         with circ.if_test((cr, 15)):
            ...
      ```
   </TabItem>
   <TabItem value="Valid" label="Valid">
      ```python
         cr = ClassicalRegister(50)
         qr = QuantumRegister(50)
         circuit = QuantumCircuit(qr, cr)
         ...
         circ.measure(qr, cr)
         with circ.if_test((cr[5], 1)):
            ...
      ```
   </TabItem>
   </Tabs>

- Nested conditionals are not allowed. For example, the following code block will not work because it has an `if_test` inside another `if_test`:
   <Tabs>
    <TabItem value="Not valid" label="Not valid">
        ```python
           c1 = ClassicalRegister(1, "c1")
           c2 = ClassicalRegister(2, "c2")
           ...
           with circ.if_test((c1, 1)):
            with circ.if_test(c2, 1)):
             ...
        ```
     </TabItem>
     <TabItem value="Valid" label="Valid">
        ```python
        cr = ClassicalRegister(2)
        ...
        with circuit.if_test((cr, 0b11)):
          ...
        ```
     </TabItem>
    </Tabs>

- Having `reset` or measurements inside conditionals is not supported.
- Arithmetic operations are not supported.
- See the [OpenQASM 3 feature table](/docs/guides/qasm-feature-table) to determine which OpenQASM 3 features are supported on Qiskit and Qiskit Runtime.
- When OpenQASM 3 (instead of `QuantumCircuit`) is used as the input format to pass circuits to Qiskit Runtime primitives, only instructions that can be loaded into Qiskit are supported. Classical operations, for example, are not supported because they cannot be loaded into Qiskit. See [Import an OpenQASM 3 program into Qiskit](/docs/guides/interoperate-qiskit-qasm3#import-an-openqasm-3-program-into-qiskit) for more information.
- The `for`, `while`, and `switch` instructions are not supported.

## Use dynamic circuits with Estimator

Since Estimator does not support dynamic circuits, you can use [Sampler](/docs/guides/get-started-with-sampler) and build your own measurement circuits instead.

To replicate Estimator's behavior, follow this process:

1. Group the terms of all observables into a partition.  This can be done by using the [`PauliList` API](/docs/api/qiskit/qiskit.quantum_info.PauliList#group_qubit_wise_commuting), for example.
     <Admonition type="note">
      You can use the [`BitArray`](https://quantum.cloud.ibm.com/docs/en/api/qiskit/qiskit.primitives.BitArray#expectation_values) primitive attribute to compute the expectation values of the provided observables.
     </Admonition>
2. Execute one basis change circuit per partition (whichever basis change needs to be done for each partition). See the Measurement bases addon utility [`measurement_bases` module](https://qiskit.github.io/qiskit-addon-utils/apidocs/qiskit_addon_utils.exp_vals.html#qiskit_addon_utils.exp_vals.get_measurement_bases) for more information. For more information, see the [documentation](https://qiskit.github.io/qiskit-addon-utils/) for the Qiskit addon utilities package.
3. Add back together the results for each partition.

## Restrictions

Review any [Feature compatibility table](/docs/guides/estimator-options#options-compatibility-table) to understand restrictions when using dynamic circuits. Note that feature compatibility is not primitive-dependent.

## Next steps

<Admonition type="tip" title="Recommendations">
- Learn how to implement accurate dynamic decoupling by using [stretch](/docs/guides/stretch).
- Review the [classical feedforward and control flow](/docs/guides/classical-feedforward-and-control-flow) guide.
- Use [circuit schedule visualization](/docs/guides/qiskit-runtime-circuit-timing) to debug and optimize your dynamic circuits.
- Not all functions are compatible with dynamic circuits. Check the feature compatibility section for [Sampler](/docs/guides/sampler-options#feature-compatibility) or [Executor](/docs/guides/executor-options#feature-compatibility) for details.
</Admonition>